##Import core libraries for filesystem handling, database access, timestamps, and password hashing

In [ ]:
import os
import sqlite3
from datetime import datetime
from pathlib import Path

from werkzeug.security import generate_password_hash


##Resolve database file path from environment variables with fallback to default location

In [ ]:
def resolve_db_path() -> Path:
    explicit_db = os.getenv("BEATSENSE_DB_PATH", "").strip()
    if explicit_db:
        return Path(explicit_db).expanduser().resolve()

    base_path = os.getenv("BEATSENSE_BASE_PATH", "").strip()
    if base_path:
        return Path(base_path).expanduser().resolve() / "beatsense.sqlite3"

    return Path.cwd().resolve() / "beatsense.sqlite3"

##Reset user-related data and recreate an admin user with a hashed password

In [ ]:
def reset_users(db_path: Path, username: str, password: str) -> None:
    db_path.parent.mkdir(parents=True, exist_ok=True)
    with sqlite3.connect(db_path) as db:
        ensure_schema(db)
        db.execute("PRAGMA foreign_keys = ON")
        db.execute("DELETE FROM password_resets")
        db.execute("DELETE FROM predictions")
        db.execute("DELETE FROM users")
        db.execute(
            "DELETE FROM sqlite_sequence WHERE name IN ('users', 'predictions', 'password_resets')"
        )
        db.execute(
            "INSERT INTO users (username, password_hash, is_admin, created_at) VALUES (?, ?, ?, ?)",
            (
                username.strip().lower(),
                generate_password_hash(password),
                1,
                datetime.utcnow().isoformat(timespec="seconds"),
            ),
        )
        db.commit()

##Ensure required database schema by creating users, predictions, and password reset tables if missing

In [ ]:
def ensure_schema(db: sqlite3.Connection) -> None:
    db.executescript(
        """
        CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT NOT NULL UNIQUE,
            password_hash TEXT NOT NULL,
            liked_genres TEXT DEFAULT '[]',
            disliked_genres TEXT DEFAULT '[]',
            is_admin INTEGER NOT NULL DEFAULT 0,
            created_at TEXT NOT NULL
        );

        CREATE TABLE IF NOT EXISTS predictions (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            user_id INTEGER NOT NULL,
            source_type TEXT NOT NULL,
            source_name TEXT NOT NULL,
            label TEXT NOT NULL,
            confidence REAL NOT NULL,
            score REAL NOT NULL,
            created_at TEXT NOT NULL,
            FOREIGN KEY(user_id) REFERENCES users(id)
        );

        CREATE TABLE IF NOT EXISTS password_resets (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            user_id INTEGER NOT NULL,
            token_hash TEXT NOT NULL UNIQUE,
            expires_at TEXT NOT NULL,
            used_at TEXT,
            created_at TEXT NOT NULL,
            FOREIGN KEY(user_id) REFERENCES users(id)
        );

        CREATE TABLE IF NOT EXISTS battles (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            created_by INTEGER,
            model_a TEXT,
            model_b TEXT,
            status TEXT DEFAULT 'pending',
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
        """
    )
    db.commit()